# Lab 01 — Next-token prediction and a tiny language model

**Free-first:** designed for Google Colab or CPU.

We will not start with a giant Transformer. We will first build the smallest useful language model so that the training objective becomes concrete.

Learning loop: **Predict → Run → Observe → Explain → Break → Decide**.


## 1. Imports
We use only PyTorch and standard Python. This is intentional: the first lab should expose the learning mechanics rather than hide them behind a trainer library.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)


## 2. Our tiny corpus
A language model needs sequences. For the first experiment, use a tiny public-domain corpus so the mechanics are visible.


In [ ]:
text = '''To be, or not to be, that is the question.
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles.
To be, or not to be.
'''.lower()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)
print('vocab size:', vocab_size)
print('tokens:', len(data))


## 3. Turn one sequence into supervised next-token examples

For a sequence like `abcd`, the training pairs are:

`a → b`, `b → c`, `c → d`.

For language-model training, this pattern is applied across many positions and batches.


In [ ]:
block_size = 16
batch_size = 32

def get_batch(batch_size=batch_size, block_size=block_size):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch()
print('x shape:', xb.shape)
print('y shape:', yb.shape)
print('first x:', decode(xb[0].tolist()))
print('first y:', decode(yb[0].tolist()))


### Checkpoint question
**Why is `y` shifted by one position?**

**Answer:** because the supervised target at position `t` is the token that follows the input token/context at position `t`.


## 4. The smallest baseline: a bigram model

Before a Transformer, use a lookup table. The model only asks:

> Given the current character/token, what token tends to come next?

This model has no attention and no hidden representation. That is exactly why it is useful.


In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

model = BigramLanguageModel(vocab_size).to(device)
logits, loss = model(xb, yb)
print('logits:', logits.shape)
print('initial loss:', float(loss))
print('theoretical random baseline ≈ log(V):', math.log(vocab_size))


## 5. Train it

Notice what is happening: we are not telling the model rules. We are repeatedly comparing its probability distribution with the observed next token and updating the embedding table.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

loss_history = []
for step in range(1000):
    xb, yb = get_batch()
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss))
    if step % 200 == 0:
        print(f'step {step:4d} | loss {loss.item():.4f}')


In [ ]:
import matplotlib.pyplot as plt
plt.plot(loss_history)
plt.xlabel('training step')
plt.ylabel('cross-entropy loss')
plt.title('Bigram training loss')
plt.show()


## 6. Generate text

The generation loop is deliberately tiny. Predict one token, append it, predict again.


In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -block_size:])
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

start = torch.tensor([[stoi['t']]], dtype=torch.long, device=device)
sample = generate(model, start, 300)[0].tolist()
print(decode(sample))


## 7. Break it on purpose

Change **one** thing at a time and predict what should happen before rerunning:

1. Set learning rate to `1.0`.
2. Train for only 10 steps.
3. Train on a corpus containing only one repeated sentence.
4. Reduce the corpus to a few characters.

Write your prediction first. Then run the experiment.


## 8. Critical-thinking answers

**Q: Why can the model generate text it never saw verbatim?**

**A:** It learns a conditional distribution rather than a simple lookup of complete sentences. Even a tiny bigram model combines learned local transition probabilities in new sequences. A Transformer can represent much richer conditional structure.

**Q: Why is this not yet an LLM?**

**A:** The model is far too small and the corpus far too limited. The point is to expose the training objective, not to pretend that scale is irrelevant.

**Q: What is the biggest limitation of the bigram model?**

**A:** It conditions only on the current token. It cannot directly model long-range dependencies such as agreement or relationships between distant words.


## 9. The bridge to Transformers

We now have the essential training loop:

`tokens → logits → cross-entropy → gradients → weight update`.

The next lectures replace the simplistic representation and context mechanism with embeddings, positional information, self-attention, multi-head attention, MLPs, residual connections, normalization, and a full Transformer block.

### Exit challenge
Explain the entire lab to another student without saying the words **Transformer**, **GPT**, or **attention**. If you can do that, you understand the core objective rather than the brand names.


## 10. Research exercise — next-token statistics on real web data

Use a real Hugging Face slice rather than the toy corpus. Sample 200 documents from `HuggingFaceFW/fineweb` (`sample-10BT`) and estimate the empirical next-character entropy. Compare it with a deliberately shuffled version of the same text. Explain what the experiment can and cannot tell you about language modeling.


In [ ]:
from datasets import load_dataset
from collections import Counter
import math, random

ds = load_dataset("HuggingFaceFW/fineweb", "sample-10BT", split="train", streaming=True)
texts = [row["text"] for _, row in zip(range(200), ds) if row.get("text")]
def entropy(text):
    pairs = Counter(zip(text[:-1], text[1:]))
    ctx = Counter(text[:-1])
    h = 0.0
    for (a,b), n in pairs.items():
        p_ab = n / sum(pairs[(a,x)] for x in set(text[text.find(a)+1:text.find(a)+2000])) if False else n / sum(v for (x,y),v in pairs.items() if x==a)
        h -= p_ab * math.log2(p_ab)
    return h

def unigram_entropy(text):
    c=Counter(text)
    n=sum(c.values())
    return -sum((v/n)*math.log2(v/n) for v in c.values())

sample="\n".join(texts)
shuffled=list(sample)
random.Random(7).shuffle(shuffled)
print("documents:",len(texts),"characters:",len(sample))
print("unigram entropy, original:",round(unigram_entropy(sample),3))
print("unigram entropy, shuffled:",round(unigram_entropy("".join(shuffled)),3))
print("same character inventory:",set(sample)==set(shuffled))


<details><summary><strong>Expected findings / answer</strong></summary>

The shuffled corpus preserves the character inventory and unigram frequencies, so its unigram entropy is essentially unchanged. What disappears is sequential structure. A next-token model exploits that structure; a bag-of-characters statistic does not. The important lesson is that **the training objective is about conditional prediction**, not merely matching token frequencies. This small experiment is intentionally a bridge from raw data to the conditional distribution $p_\\theta(x_t\\mid x_{<t})$.

Do not claim that character entropy is model perplexity: the tokenization, context length, architecture, and trained parameters are different variables.
</details>
